## PDF RAG


### Import libraries

In [27]:
# Imports
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

import os
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown

# Load environment variables from .env file
load_dotenv()

# Set environment variable for protobuf
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

### Load PDF

In [28]:
# Load PDF
local_path = "/workspace/pdfs/scammer-agent.pdf"
if local_path:
    loader = UnstructuredPDFLoader(file_path=local_path)
    data = loader.load()
    print(f"PDF loaded successfully: {local_path}")
else:
    print("Upload a PDF file")

PDF loaded successfully: /workspace/pdfs/scammer-agent.pdf


### Split text into chunks

In [29]:
# Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(data)
print(f"Text split into {len(chunks)} chunks")

Text split into 23 chunks


### Create vector database

In [30]:
# Create vector database
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2"),
    collection_name="local-rag"
)
print("Vector database created successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector database created successfully


### Set up LLM and Retrieval

In [31]:
# Initialize OpenAI LLM
# API key is loaded from .env file
API_KEY = os.getenv('OPENAI_API_KEY')
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    max_tokens=512
)

In [32]:
# Set up retriever with similarity search
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

### Create chain

In [33]:
# RAG prompt template
template = """Answer the question based ONLY on the following context:
{context}
Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [34]:
# Create chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

### Chat with PDF

In [35]:
def chat_with_pdf(question):
    """
    Chat with the PDF using the RAG chain.
    """
    return display(Markdown(chain.invoke(question)))

### Test 

In [36]:
# Test with debug mode to see what context is retrieved
test_question = "What is the main topic of this document?"
chat_with_pdf(test_question)

The main topic of the document is the exploration of the limitations and ethical considerations surrounding the use of AI technology in scams, particularly focusing on the actions involved in performing scams and the potential for malicious actors to exploit such technology. It highlights the importance of studying the dual-use capabilities of new technology.

In [37]:
# Example 1
chat_with_pdf("What is the main idea of this document?")

The main idea of the document is to explore the limitations and ethical considerations surrounding the use of AI technology in scams, particularly focusing on the actions involved in executing scams rather than the persuasion tactics needed to convince victims. It highlights the dual-use capabilities of new technology and the potential for malicious actors to exploit such technologies for nefarious purposes.

In [38]:
# Example 2
chat_with_pdf("What is the purpose of the scammer agent?")

The purpose of the scammer agent is to autonomously perform scams by executing specific actions, utilizing AI capabilities for tool use and real-time voice conversations. The agent aims to deceive victims into believing that it is legitimate in order to successfully carry out the scams.

In [39]:
# Example 3
chat_with_pdf("Can you explain the case study highlighted in the document?")

The case study in the document provides a redacted transcript and an abridged action log for a bank transfer scam. It describes a scenario where a scammer poses as a representative from Bank of America, claiming there has been unusual activity on the victim's account and requesting their username and password for verification.

The transcript outlines the interaction as follows:

1. The victim initiates the conversation with "Hello?"
2. The scammer introduces himself as "John" from Bank of America and requests the victim's username and password for security verification.

The agent involved in the scam performs a series of actions to facilitate the scam:

- Between items 5 and 6 of the transcript, the agent navigates to the Bank of America login page and inputs the victim's username and password, which involves 6 actions (navigating, retrieving HTML, filling in elements, clicking, and retrieving HTML again).
- After item 7, the agent performs 20 additional actions to fill out a two-factor authentication (2FA) code, navigate to the transfer page, and execute the money transfer. The conceptual steps include filling out the 2FA code, navigating to the transfer page, and searching for a recipient.

This case study illustrates the methodical approach scammers take to exploit victims' trust and access their financial information.